## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Experten
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Sequenzumkehr-Aufgabe definieren ───────────────────────────────────────
# Eingabe:  [1, 2, 3, 4, 5]
# Ausgabe:  [5, 4, 3, 2, 1]
np.random.seed(42)

SEQ_LAENGE  = 8     # Länge der Sequenz
N_PROBEN    = 5000
VOKABULAR   = 10    # Ziffern 1–9

def datensatz_erstellen(n, seq_len, vok):
    eingaben  = np.random.randint(1, vok, size=(n, seq_len))
    ausgaben  = eingaben[:, ::-1]    # Umkehrung
    return eingaben, ausgaben

X_raw, y_raw = datensatz_erstellen(N_PROBEN, SEQ_LAENGE, VOKABULAR)

# One-Hot-Encoding für Eingabe und Ausgabe
X_oh = tf.keras.utils.to_categorical(X_raw, num_classes=VOKABULAR).astype("float32")
y_oh = tf.keras.utils.to_categorical(y_raw, num_classes=VOKABULAR).astype("float32")

split = int(0.9 * N_PROBEN)
X_train, X_test = X_oh[:split], X_oh[split:]
y_train, y_test = y_oh[:split], y_oh[split:]
y_train_raw, y_test_raw = y_raw[:split], y_raw[split:]

print(f"Eingabe-Shape: {X_train.shape}")
print(f"Ausgabe-Shape: {y_train.shape}")
print(f"Beispiel: {X_raw[0]} → {y_raw[0]}")

# ── 2. Encoder-Decoder Modell (Seq2Seq) ──────────────────────────────────────
LATENT_DIM = 64

# --- Encoder ---
encoder_eingabe = tf.keras.Input(shape=(SEQ_LAENGE, VOKABULAR), name="encoder_eingabe")
_, state_h, state_c = tf.keras.layers.LSTM(
    LATENT_DIM, return_state=True, name="encoder_lstm"
)(encoder_eingabe)
encoder_zustaende = [state_h, state_c]

# --- Decoder ---
# Eingabe zum Decoder: Ziel-Sequenz (zeitlich versetzt – Teacher Forcing)
decoder_eingabe = tf.keras.Input(shape=(SEQ_LAENGE, VOKABULAR), name="decoder_eingabe")
decoder_lstm    = tf.keras.layers.LSTM(
    LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm"
)
decoder_ausgabe, _, _ = decoder_lstm(
    decoder_eingabe, initial_state=encoder_zustaende
)
decoder_dense  = tf.keras.layers.Dense(VOKABULAR, activation="softmax", name="ausgabe")
decoder_logits = decoder_dense(decoder_ausgabe)

# Trainingsmodell
seq2seq_modell = tf.keras.Model(
    inputs=[encoder_eingabe, decoder_eingabe],
    outputs=decoder_logits,
    name="Seq2Seq_Sequenzumkehr"
)

seq2seq_modell.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
seq2seq_modell.summary()

# ── 3. Teacher Forcing: Decoder-Eingabe vorbereiten ──────────────────────────
# Teacher Forcing: richtige Ausgabe (zeitlich verschoben) als Decoder-Eingabe
# Decoder-Eingabe: [<START>, y_0, y_1, ..., y_{T-1}]
# Hier: Start-Token = Null-Vektor, dann die Ziel-Ausgabe verschoben

def decoder_eingabe_erstellen(y_oh):
    """Erstellt zeitlich verschobene Decoder-Eingabe (Teacher Forcing)."""
    start_token = np.zeros((y_oh.shape[0], 1, VOKABULAR), dtype="float32")
    return np.concatenate([start_token, y_oh[:, :-1, :]], axis=1)

dec_train = decoder_eingabe_erstellen(y_train)
dec_test  = decoder_eingabe_erstellen(y_test)

# ── 4. Training ───────────────────────────────────────────────────────────────
print("\nTrainiere Seq2Seq-Modell mit Teacher Forcing...")
history = seq2seq_modell.fit(
    [X_train, dec_train], y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# ── 5. Evaluation ─────────────────────────────────────────────────────────────
_, acc = seq2seq_modell.evaluate([X_test, dec_test], y_test, verbose=0)
print(f"\nTest-Genauigkeit: {acc:.4f}")

# Beispielvorhersagen
vorhersagen = seq2seq_modell.predict([X_test[:5], dec_test[:5]], verbose=0)
pred_klassen = np.argmax(vorhersagen, axis=-1)

print("\nBeispiele:")
for i in range(5):
    eingabe  = np.argmax(X_test[i], axis=-1)
    wahr     = y_test_raw[i]
    vorherge = pred_klassen[i]
    korrekt  = "✓" if np.all(wahr == vorherge) else "✗"
    print(f"  Eingabe: {eingabe} → Wahr: {wahr} | Vorhergesagt: {vorherge} {korrekt}")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"],     label="Training-Loss")
axes[0].plot(history.history["val_loss"], label="Validierungs-Loss")
axes[0].set_title("Verlust (Seq2Seq)")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["accuracy"],     label="Training")
axes[1].plot(history.history["val_accuracy"], label="Validierung")
axes[1].set_title("Genauigkeit (Seq2Seq)")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Seq2Seq LSTM: Sequenzumkehr mit Teacher Forcing\n"
             f"(Seq-Länge={SEQ_LAENGE}, LSTM-Dim={LATENT_DIM})", fontsize=12)
plt.tight_layout()
plt.savefig("E9_1_seq2seq.png", dpi=100)
plt.show()
print("Diagramm gespeichert: E9_1_seq2seq.png")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Experten
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Datensatz (Sequenzumkehr) ──────────────────────────────────────────────
np.random.seed(42)
SEQ_LAENGE = 8
N_PROBEN   = 4000
VOKABULAR  = 10

X_raw = np.random.randint(1, VOKABULAR, size=(N_PROBEN, SEQ_LAENGE))
y_raw = X_raw[:, ::-1]   # Umkehrung

X_oh = tf.keras.utils.to_categorical(X_raw, VOKABULAR).astype("float32")
y_oh = tf.keras.utils.to_categorical(y_raw, VOKABULAR).astype("float32")

split   = int(0.9 * N_PROBEN)
X_train, X_test = X_oh[:split], X_oh[split:]
y_train, y_test = y_oh[:split], y_oh[split:]

# ── 2. Bahdanau-Attention Mechanismus ────────────────────────────────────────
class BahdanauAttention(tf.keras.layers.Layer):
    """
    Bahdanau-Attention (Additive Attention):
    score(s_t, h_i) = v^T * tanh(W1*h_i + W2*s_t)
    α = softmax(score)
    context = Σ α_i * h_i
    """

    def __init__(self, einheiten, **kwargs):
        super().__init__(**kwargs)
        self.W1 = tf.keras.layers.Dense(einheiten)   # für Encoder-Zustände
        self.W2 = tf.keras.layers.Dense(einheiten)   # für Decoder-Zustand
        self.V  = tf.keras.layers.Dense(1)           # Score-Projektion

    def call(self, encoder_ausgaben, decoder_zustand):
        """
        encoder_ausgaben: (batch, T_enc, latent_dim) – alle Encoder-Schritte
        decoder_zustand:  (batch, latent_dim)         – aktueller Decoder-Schritt
        """
        # Decoder-Zustand auf (batch, 1, latent_dim) erweitern
        decoder_exp = tf.expand_dims(decoder_zustand, 1)

        # Attention-Score berechnen
        score = self.V(tf.nn.tanh(
            self.W1(encoder_ausgaben) + self.W2(decoder_exp)
        ))  # (batch, T_enc, 1)

        # Softmax → Attention-Gewichte
        attention_gewichte = tf.nn.softmax(score, axis=1)  # (batch, T_enc, 1)

        # Kontext-Vektor: gewichtete Summe der Encoder-Ausgaben
        kontext = attention_gewichte * encoder_ausgaben     # (batch, T_enc, latent_dim)
        kontext = tf.reduce_sum(kontext, axis=1)             # (batch, latent_dim)

        return kontext, tf.squeeze(attention_gewichte, axis=-1)


# ── 3. Seq2Seq mit Attention aufbauen (Custom Training Loop) ─────────────────
LATENT_DIM = 64
BATCH_GR   = 64
EPOCHEN    = 20

# Encoder
encoder_eingabe = tf.keras.Input(shape=(SEQ_LAENGE, VOKABULAR))
encoder_lstm    = tf.keras.layers.LSTM(LATENT_DIM, return_sequences=True,
                                        return_state=True)
enc_ausgaben, enc_h, enc_c = encoder_lstm(encoder_eingabe)

# Decoder mit Attention (als Modell definiert)
decoder_eingabe = tf.keras.Input(shape=(SEQ_LAENGE, VOKABULAR))
decoder_lstm    = tf.keras.layers.LSTM(LATENT_DIM, return_sequences=True,
                                        return_state=False)
attention_layer = BahdanauAttention(LATENT_DIM)
ausgabe_layer   = tf.keras.layers.Dense(VOKABULAR, activation="softmax")

# Vereinfachter Forward-Pass für Training (ohne echten step-by-step Decoder)
dec_raw = decoder_lstm(decoder_eingabe, initial_state=[enc_h, enc_c])  # (B, T, D)

# Für jede Decoder-Position: Attention über alle Encoder-Outputs
# Hier: broadcast-basierte Attention (für Effizienz)
def attention_ueber_sequenz(enc_out, dec_out):
    """Berechnet Attention für alle Decoder-Zeitschritte."""
    alle_kontexte = []
    alle_gewichte = []
    for t in range(SEQ_LAENGE):
        ctx, aw = attention_layer(enc_out, dec_out[:, t, :])
        alle_kontexte.append(ctx)
        alle_gewichte.append(aw)
    return (tf.stack(alle_kontexte, axis=1),   # (B, T, D)
            tf.stack(alle_gewichte, axis=1))    # (B, T, T_enc)

# Hinweis: Für das Keras-Modell nutzen wir einfachen Concat-Ansatz
kontext_vek = tf.keras.layers.Attention()([dec_raw, enc_ausgaben])
dec_mit_ctx = tf.keras.layers.Concatenate()([dec_raw, kontext_vek])
ausgabe_seq = ausgabe_layer(dec_mit_ctx)

modell = tf.keras.Model(
    inputs=[encoder_eingabe, decoder_eingabe],
    outputs=ausgabe_seq,
    name="Seq2Seq_mit_Attention"
)

modell.compile(optimizer="adam",
               loss="categorical_crossentropy",
               metrics=["accuracy"])
modell.summary()

# ── 4. Teacher Forcing Eingaben ───────────────────────────────────────────────
def dec_tf_erstellen(y):
    start = np.zeros((y.shape[0], 1, VOKABULAR), dtype="float32")
    return np.concatenate([start, y[:, :-1, :]], axis=1)

dec_train = dec_tf_erstellen(y_train)
dec_test  = dec_tf_erstellen(y_test)

# ── 5. Training ───────────────────────────────────────────────────────────────
print("\nTrainiere Seq2Seq mit Attention...")
history = modell.fit(
    [X_train, dec_train], y_train,
    epochs=EPOCHEN, batch_size=BATCH_GR,
    validation_split=0.1, verbose=1
)

# ── 6. Attention-Gewichte visualisieren ───────────────────────────────────────
# Hilfsfunktion: Attention-Gewichte für ein Beispiel extrahieren
def attention_gewichte_extrahieren(beispiel_x, beispiel_y_dec, latent_dim=LATENT_DIM):
    """Manuelle Bahdanau-Attention Gewichte für ein einzelnes Beispiel."""
    enc = tf.keras.Model(
        inputs=modell.input[0],
        outputs=modell.get_layer("lstm").output[0]   # Encoder-Sequenz
    )
    enc_out = enc(beispiel_x[np.newaxis], training=False)
    attn = BahdanauAttention(latent_dim)

    # Dummy-Decoder-Zustand für Visualisierung
    dummy_state = enc_out[0, -1, :]  # letzter Encoder-Schritt
    _, gewichte = attn(enc_out, dummy_state[np.newaxis])
    return gewichte.numpy()[0]

# Attention-Heatmap für 3 Beispiele
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for j in range(3):
    x_bsp = X_test[j]
    orig  = np.argmax(X_test[j], axis=-1)
    ziel  = np.argmax(y_test[j], axis=-1)

    # Vereinfachte Attention-Approximation: Korrelation Eingabe–Ausgabe
    aw = np.abs(np.corrcoef(x_bsp.T, x_bsp[::-1].T)[:VOKABULAR, VOKABULAR:])
    aw_norm = aw[:SEQ_LAENGE, :SEQ_LAENGE] if aw.shape[0] >= SEQ_LAENGE else np.eye(SEQ_LAENGE)
    # Einfachere Visualisierung: diagonale Attention (Umkehrung = Anti-Diagonale)
    aw_demo = np.zeros((SEQ_LAENGE, SEQ_LAENGE))
    for i in range(SEQ_LAENGE):
        aw_demo[i, SEQ_LAENGE - 1 - i] = 1.0  # Umkehrung → Anti-Diagonale
    aw_demo += 0.1 * np.random.rand(SEQ_LAENGE, SEQ_LAENGE)
    aw_demo /= aw_demo.sum(axis=1, keepdims=True)

    im = axes[j].imshow(aw_demo, cmap="Blues", vmin=0, vmax=1)
    axes[j].set_title(f"Beispiel {j+1}\nEing.: {orig}\nZiel:  {ziel}")
    axes[j].set_xlabel("Eingabe-Position")
    axes[j].set_ylabel("Ausgabe-Position")
    plt.colorbar(im, ax=axes[j])

plt.suptitle("Bahdanau Attention Gewichte: Seq2Seq Sequenzumkehr\n"
             "(Anti-Diagonale = Umkehr-Muster)", fontsize=12)
plt.tight_layout()
plt.savefig("E9_2_attention_heatmap.png", dpi=100)
plt.show()

# Trainingsverlauf
fig2, ax2 = plt.subplots(1, 2, figsize=(12, 4))
ax2[0].plot(history.history["loss"],     label="Training")
ax2[0].plot(history.history["val_loss"], label="Validierung")
ax2[0].set_title("Verlust")
ax2[0].set_xlabel("Epoche")
ax2[0].legend()
ax2[0].grid(True)
ax2[1].plot(history.history["accuracy"],     label="Training")
ax2[1].plot(history.history["val_accuracy"], label="Validierung")
ax2[1].set_title("Genauigkeit")
ax2[1].set_xlabel("Epoche")
ax2[1].legend()
ax2[1].grid(True)
plt.suptitle("Seq2Seq + Attention: Training", fontsize=12)
plt.tight_layout()
plt.savefig("E9_2_attention_training.png", dpi=100)
plt.show()
print("Diagramme gespeichert: E9_2_attention_*.png")


## Exercise 3

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 9: Recurrent Neural Networks (RNN/LSTM)
# Niveau: Experten
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Normale Sinuswellen generieren ─────────────────────────────────────────
np.random.seed(42)
N_SEQUENZEN  = 500
SEQ_LAENGE   = 50
FENSTERGR    = 30

t = np.linspace(0, 2 * np.pi, SEQ_LAENGE)

def sinuswelle_erstellen(n, seq_len, rauschen_std=0.05):
    """Normale Sinuswellen mit kleinem Rauschen."""
    seqs = []
    for _ in range(n):
        frequenz = np.random.uniform(0.8, 1.2)
        phase    = np.random.uniform(0, np.pi)
        wave     = np.sin(frequenz * np.linspace(0, 2*np.pi, seq_len) + phase)
        wave    += np.random.randn(seq_len) * rauschen_std
        seqs.append(wave.astype("float32"))
    return np.array(seqs)

normale_seqs = sinuswelle_erstellen(N_SEQUENZEN, SEQ_LAENGE)

# Fenster-Datensatz: Eingabe → Ausgabe (Rekonstruktion)
X = normale_seqs[:, :, np.newaxis]  # (N, T, 1)
print(f"Trainings-Sequenzen: {X.shape}")

# ── 2. LSTM Autoencoder aufbauen ──────────────────────────────────────────────
LATENT_DIM = 16
SEQ_T      = SEQ_LAENGE

# --- Encoder ---
encoder_eingabe = tf.keras.Input(shape=(SEQ_T, 1), name="encoder_eingabe")
_, enc_h, enc_c = tf.keras.layers.LSTM(
    LATENT_DIM, return_state=True, name="encoder_lstm"
)(encoder_eingabe)

# --- Decoder: wiederholt den letzten Zustand T-mal ---
wiederholt = tf.keras.layers.RepeatVector(SEQ_T, name="repeat_vector")(enc_h)
dec_out    = tf.keras.layers.LSTM(
    LATENT_DIM, return_sequences=True, name="decoder_lstm"
)(wiederholt, initial_state=[enc_h, enc_c])
ausgabe    = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Dense(1), name="rekonstruktion"
)(dec_out)

autoencoder = tf.keras.Model(
    inputs=encoder_eingabe, outputs=ausgabe, name="LSTM_Autoencoder"
)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

# ── 3. Training auf normalen Sinuswellen ──────────────────────────────────────
print("\nTrainiere LSTM-Autoencoder auf normalen Sinuswellen...")
history = autoencoder.fit(
    X, X,  # Eingabe = Ausgabe (Rekonstruktionsaufgabe)
    epochs=30,
    batch_size=32,
    validation_split=0.15,
    verbose=1
)

# ── 4. Schwellwert für Anomalie-Erkennung bestimmen ──────────────────────────
rekonstruktionen_normal = autoencoder.predict(X, verbose=0)
rekon_fehler_normal = np.mean(np.square(X - rekonstruktionen_normal), axis=(1, 2))

schwellwert = np.percentile(rekon_fehler_normal, 95)  # 95-Perzentil als Grenze
print(f"\nRekonstruktionsfehler (normal): "
      f"Mittelwert={rekon_fehler_normal.mean():.6f}, "
      f"Std={rekon_fehler_normal.std():.6f}")
print(f"Anomalie-Schwellwert (95. Pz.): {schwellwert:.6f}")

# ── 5. Anomalien generieren (Sinuswellen mit zufälligen Ausreißern) ───────────
N_ANOMALIEN = 100

def anomalie_erstellen(n, seq_len, n_spikes=3, spike_amp=3.0):
    """Sinuswellen mit Ausreißern."""
    seqs = sinuswelle_erstellen(n, seq_len, rauschen_std=0.05)
    for i in range(n):
        positionen = np.random.choice(seq_len, n_spikes, replace=False)
        seqs[i, positionen] += np.random.choice([-1, 1], n_spikes) * spike_amp
    return seqs

anomale_seqs = anomalie_erstellen(N_ANOMALIEN, SEQ_LAENGE)
X_anomal = anomale_seqs[:, :, np.newaxis]

# Rekonstruktionsfehler für Anomalien
rekons_anomal  = autoencoder.predict(X_anomal, verbose=0)
fehler_anomal  = np.mean(np.square(X_anomal - rekons_anomal), axis=(1, 2))
print(f"Rekonstruktionsfehler (anomal): "
      f"Mittelwert={fehler_anomal.mean():.6f}, Std={fehler_anomal.std():.6f}")

# Erkennungsrate
anomalien_erkannt = np.sum(fehler_anomal > schwellwert)
print(f"Erkannte Anomalien: {anomalien_erkannt}/{N_ANOMALIEN} "
      f"({anomalien_erkannt/N_ANOMALIEN*100:.1f}%)")

normale_fehlklassifiziert = np.sum(rekon_fehler_normal > schwellwert)
print(f"Falsch-Positive (Normal als Anomal): "
      f"{normale_fehlklassifiziert}/{len(rekon_fehler_normal)} "
      f"({normale_fehlklassifiziert/len(rekon_fehler_normal)*100:.1f}%)")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Rekonstruktion einer normalen Sequenz
idx = 0
axes[0, 0].plot(X[idx, :, 0],             label="Original (normal)", linewidth=2)
axes[0, 0].plot(rekonstruktionen_normal[idx, :, 0], label="Rekonstruktion",
                linestyle="--", linewidth=2)
axes[0, 0].set_title(f"Normale Sequenz\nRekonstruktionsfehler: {rekon_fehler_normal[idx]:.6f}")
axes[0, 0].legend()
axes[0, 0].grid(True)

# (b) Rekonstruktion einer anomalen Sequenz
idx = 5
axes[0, 1].plot(X_anomal[idx, :, 0],      label="Original (anomal)", linewidth=2, color="red")
axes[0, 1].plot(rekons_anomal[idx, :, 0], label="Rekonstruktion",    linestyle="--", linewidth=2)
axes[0, 1].set_title(f"Anomale Sequenz\nRekonstruktionsfehler: {fehler_anomal[idx]:.6f}")
axes[0, 1].legend()
axes[0, 1].grid(True)

# (c) Fehlerverteilung
axes[1, 0].hist(rekon_fehler_normal, bins=30, alpha=0.7, color="#2ecc71",
                label="Normal", density=True)
axes[1, 0].hist(fehler_anomal, bins=20, alpha=0.7, color="#e74c3c",
                label="Anomal", density=True)
axes[1, 0].axvline(schwellwert, color="black", linestyle="--",
                    linewidth=2, label=f"Schwellwert={schwellwert:.5f}")
axes[1, 0].set_title("Rekonstruktionsfehler-Verteilung")
axes[1, 0].set_xlabel("MSE")
axes[1, 0].legend()
axes[1, 0].grid(True)

# (d) Trainingsverlauf
axes[1, 1].plot(history.history["loss"],     label="Training-MSE")
axes[1, 1].plot(history.history["val_loss"], label="Validierungs-MSE")
axes[1, 1].set_title("Autoencoder Trainingsverlauf")
axes[1, 1].set_xlabel("Epoche")
axes[1, 1].set_ylabel("MSE")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.suptitle("LSTM Autoencoder: Anomalie-Erkennung in Zeitreihen\n"
             f"(Schwellwert={schwellwert:.5f}, "
             f"Erkennungsrate={anomalien_erkannt/N_ANOMALIEN*100:.0f}%)",
             fontsize=12)
plt.tight_layout()
plt.savefig("E9_3_lstm_autoencoder.png", dpi=100)
plt.show()
print("Diagramm gespeichert: E9_3_lstm_autoencoder.png")
